# Exp 1 — Vanilla ViT (no SPT, no LSA, no distillation)

DeiT-tiny: depth=12, dim=192, heads=3, patch_size=16, img_size=128  
Augmentations: RandAugment + ElasticTransform + repeated aug (N=3)  
Training: 50 epochs, 15 warmup, cosine LR, stochastic depth=0.1

In [ ]:
# ── Setup: clone repo and install deps ─────────────────────────────────────
import os, subprocess, sys, importlib.util

REPO_URL = "https://github.com/ArjunS07/cs-148"   # TODO: set your git remote URL
REPO_DIR = "/content/repo"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch", "torchvision", "huggingface-hub",
                    "matplotlib", "seaborn", "scikit-learn"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(os.path.dirname("."), "..", ".."))

P3_SRC = os.path.join(REPO_DIR, "project3", "src")
P2_SRC = os.path.join(REPO_DIR, "project2", "src")

# Keep project3 ahead of project2 for plain imports (e.g., import train, import model)
for p in [P2_SRC, P3_SRC]:
    while p in sys.path:
        sys.path.remove(p)
for p in [P2_SRC, P3_SRC]:
    sys.path.insert(0, p)  # final order: P3_SRC, then P2_SRC

def _load_module(module_name, module_path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

# Explicit aliases so project2 and project3 models can be used side-by-side
P3_MODEL = _load_module("p3_model", os.path.join(P3_SRC, "model.py"))
P2_MODEL = _load_module("p2_model", os.path.join(P2_SRC, "model.py"))

print("Python path set up.")
print("  P3_SRC:", P3_SRC)
print("  P2_SRC:", P2_SRC)
print("  sys.path priority:", sys.path[:3])

Python path set up. P3_SRC: /content/repo/project3/src


In [17]:
# ── Data path ───────────────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = os.path.join(
        "/content/drive/MyDrive",
        "College",
        "Year 2 - Sophomore ",
        "Y2 Quarter 2",
        "cs148",
        "dataset"
    )
else:
    DATA_DIR = os.path.join(REPO_DIR, "project2", "data", "dataset")

print("Data dir:", DATA_DIR)
print("Exists:", os.path.exists(DATA_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data dir: /content/drive/MyDrive/College/Year 2 - Sophomore /Y2 Quarter 2/cs148/dataset
Exists: True


In [18]:
# ── GPU check ───────────────────────────────────────────────────────────────
import torch
device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [ ]:
# ── Experiment config ───────────────────────────────────────────────────────
import argparse

SAVE_DIR = "/content/checkpoints/exp1_vanilla" if IN_COLAB else "../checkpoints/exp1_vanilla"

args = argparse.Namespace(
    data_dir=DATA_DIR,
    save_dir=SAVE_DIR,
    # Training
    epochs=50,
    batch_size=64,
    lr=1e-3,
    weight_decay=0.05,
    warmup_epochs=15,
    drop_path_rate=0.1,
    img_size=128,
    repeat_aug=3,
    patience=0,
    num_workers=2,
    seed=42,
    # Dataset
    synthetic_n=3000,
    val_fraction=0.15,
    # Model: vanilla (no SPT, no LSA, no distillation)
    use_spt=False,
    use_lsa=False,
    distillation="none",
    teacher_path=os.path.join(REPO_DIR, "project2", "checkpoints", "run10_final_9k", "best_model.pt"),
    tau=4.0,
    lambda_dist=0.5,
)

import numpy as np
torch.manual_seed(args.seed)
np.random.seed(args.seed)
print("Config ready.")

In [ ]:
# ── Sanity check: model architecture ────────────────────────────────────────
model = P3_MODEL.build_model(use_spt=args.use_spt, use_lsa=args.use_lsa, use_dist_token=False)
n = P3_MODEL.count_parameters(model)
print(f"ViT (vanilla): {n:,} params ({n/1e6:.2f}M)")

x = torch.randn(2, 3, 128, 128)
out = model(x)
print(f"Forward pass: input {x.shape} -> output {out.shape}")

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
import train
train.train(args)

In [ ]:
# ── Download results (Colab only) ────────────────────────────────────────────
if IN_COLAB:
    import shutil
    zip_path = "/content/exp1_vanilla_results.zip"
    shutil.make_archive("/content/exp1_vanilla_results", "zip", SAVE_DIR)
    from google.colab import files
    files.download(zip_path)